# Lab 5.1 &mdash; LangChain: rebuild the AskOps agent

**Time:** about 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph**

### What you will do
- Join a prompt, a model and a parser with the pipe `|`
- Turn the AskOps functions into tools, and read the schema LangChain builds
- Build the whole agent with one call to `create_agent`
- Stop an agent that will not stop, with `recursion_limit`
- Run it on the sandbox model and compare it with your Day 1 agent

> **How this lab works.** Fill in every `BLANK`, then run the **Self-check** cell under each
> section. It prints `[PASS]`, `[FAIL]` or `[TODO]` for each check. Graded cells never call the
> sandbox model, so your score does not depend on it. Cells marked **Run it for real** do call
> the model. If it is not reachable, they print how to fix it instead of crashing.

> **The same agent as Module 4.** On Day 1 you wrote the loop by hand in `agent.py`.
> Here LangChain writes it. Keep your Lab 4.4 notes open: you compare the numbers at the end.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, copy, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "aac-lab-5-1")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one check. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then run this cell again)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# The sandbox already has a model set up: nothing to install, no key to enter.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("No model is set up here. In a sandbox terminal run `env | grep LAB_LLM`.")
        print("If it prints nothing, tell your trainer. The graded cells still work.")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model that talks to the sandbox model."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One call to the model. Returns text, or an error string. Never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not set up -- the graded cells still work)")

In [ ]:
# ------------------------------------------------- AskOps: the same data and tools as Module 4
# Six runbooks and three open incidents. Nothing here is real, and nothing leaves this notebook.
RUNBOOKS = [{'id': 'RB-101',
  'title': 'Payments API returns 502 after deploy',
  'service': 'payments',
  'tags': ['deploy', 'rollback', '502', 'gateway'],
  'steps': ['Check the deploy pipeline for the last release',
            'Compare error rate before and after the release',
            'Roll back with the release tool if the error rate doubled',
            'Open an incident if rollback does not clear it']},
 {'id': 'RB-102',
  'title': 'Database connection pool exhausted',
  'service': 'payments',
  'tags': ['database', 'pool', 'timeout', 'connections'],
  'steps': ['Confirm the pool metric is at its maximum',
            'Find long-running queries and their owners',
            'Raise the pool size only as a temporary measure',
            'File a ticket for the query that held connections']},
 {'id': 'RB-201',
  'title': 'Login latency above 2 seconds',
  'service': 'auth',
  'tags': ['latency', 'login', 'cache', 'slow'],
  'steps': ['Check the token cache hit rate',
            'Warm the cache if a node restarted',
            'Scale the auth service if CPU is above 80 percent']},
 {'id': 'RB-202',
  'title': 'Certificate expiring within 7 days',
  'service': 'auth',
  'tags': ['tls', 'certificate', 'expiry'],
  'steps': ['List certificates expiring this week',
            'Request renewal from the PKI portal',
            'Deploy the renewed certificate and verify the chain']},
 {'id': 'RB-301',
  'title': 'Nightly batch job did not finish',
  'service': 'reporting',
  'tags': ['batch', 'job', 'timeout', 'retry'],
  'steps': ['Read the job log for the last completed step',
            'Re-run from the failed step, not from the start',
            'Tell report consumers the expected delay']},
 {'id': 'RB-302',
  'title': 'Disk usage above 90 percent on report nodes',
  'service': 'reporting',
  'tags': ['disk', 'storage', 'cleanup'],
  'steps': ['Find the largest directories',
            'Delete report archives older than 30 days',
            'Add a retention rule so it does not recur']}]

INCIDENTS_AT_START = [{'id': 'INC-9001',
  'title': 'Payments 502s after 14:00 release',
  'severity': 'high',
  'service': 'payments',
  'opened_at': '2026-09-14T14:07:00',
  'runbook_id': 'RB-101'},
 {'id': 'INC-9002',
  'title': 'Slow logins in the morning peak',
  'severity': 'medium',
  'service': 'auth',
  'opened_at': '2026-09-15T09:12:00',
  'runbook_id': 'RB-201'},
 {'id': 'INC-9003',
  'title': 'Batch report late for finance',
  'severity': 'low',
  'service': 'reporting',
  'opened_at': '2026-09-16T06:30:00',
  'runbook_id': None}]
INCIDENTS = copy.deepcopy(INCIDENTS_AT_START)

def reset_data():
    """Put the incident list back as it started. The checks call this, so they leave no trace."""
    INCIDENTS[:] = copy.deepcopy(INCIDENTS_AT_START)

# The four AskOps tools from labs/module-4-agent/agent.py, as plain functions.
def search_runbooks(query, service=None):
    words = {w for w in query.lower().split() if len(w) >= 3}
    hits = [{"id": r["id"], "title": r["title"]} for r in RUNBOOKS
            if service in (None, r["service"])
            and words & set(r["title"].lower().split() + r["tags"])]
    return json.dumps(hits[:3])

def get_runbook(runbook_id):
    for r in RUNBOOKS:
        if r["id"] == runbook_id:
            return json.dumps(r)
    return f"ERROR: no runbook {runbook_id}. Use search_runbooks first."

def list_incidents():
    return json.dumps(INCIDENTS)

def open_incident(title, severity, service, runbook_id=None):
    """The only tool that WRITES. It adds a record that other people see."""
    incident = {"id": f"INC-{9001 + len(INCIDENTS)}", "title": title,
                "severity": severity, "service": service, "runbook_id": runbook_id}
    INCIDENTS.insert(0, incident)
    return json.dumps(incident)

print(len(RUNBOOKS), "runbooks,", len(INCIDENTS), "open incidents")

In [ ]:
# ------------------------------------------------- a fake model for the graded cells
# It is not an AI. It replays a script of replies, so the checks give the same result every time.
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
import types

class FakeModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):     # a real model sends the tools; the fake ignores them
        return self

def fake(*replies):
    """A fake model that gives these replies, in order. A reply is text, or a tool call."""
    return FakeModel(messages=iter([r if isinstance(r, AIMessage) else AIMessage(r) for r in replies]))

def call(tool, n=1, **args):
    """A scripted tool call, the same shape a real model sends."""
    return AIMessage("", tool_calls=[{"name": tool, "args": args, "id": f"call_{n}"}])

# The plain Module 4 functions, kept under one name so the tools below can call them.
askops = types.SimpleNamespace(search_runbooks=search_runbooks, get_runbook=get_runbook,
                               list_incidents=list_incidents, open_incident=open_incident)

## Concept

Your Day 1 agent did four jobs by hand. LangChain has a ready-made part for each one.

| Day 1, in `agent.py` | LangChain part |
|---|---|
| Build the list of messages | **Prompt template** (`ChatPromptTemplate`) |
| `chat()`: one HTTP POST to the model | **Chat model** (`ChatOpenAI`) |
| Dig the text out of the JSON reply | **Output parser** (`StrOutputParser`) |
| `TOOL_SPECS` and `TOOLS[name](**args)` | **Tool** (`@tool`) |
| The `for` loop and `MAX_STEPS` | **Agent** (`create_agent`) and `recursion_limit` |

The messages that go to the model are the same as on Day 1. Only the amount of code you write changes.

## Section 1 &mdash; A chain: `prompt | model | parser`

The pipe `|` passes the output of one part into the next. The prompt fills its blanks and returns
messages. The model returns an `AIMessage`. The parser takes out the text and returns a plain string.
A fixed set of steps like this, with no tools, is a **chain**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an SRE. Answer in one line."),
    ("user", "Summarise this incident: {text}"),
])

def summariser(model):
    """A chain that fills the prompt, calls the model and returns plain text."""
    return BLANK        # TODO: join prompt, model and StrOutputParser() with the pipe |

In [ ]:
# --- Self-check: Section 1
check("the prompt fills the {text} blank",
      lambda: "502 after deploy" in prompt.invoke({"text": "502 after deploy"}).to_messages()[1].content)
check("the chain returns plain text, not a message object",
      lambda: isinstance(summariser(fake("Payments: 502 since the deploy.")).invoke({"text": "x"}), str),
      "the parser at the end turns the AIMessage into a str")
check("the chain returns the model's words",
      lambda: summariser(fake("Payments: 502 since the deploy.")).invoke({"text": "x"})
              == "Payments: 502 since the deploy.")
check("batch runs two inputs and returns two answers",
      lambda: sorted(summariser(fake("one", "two")).batch([{"text": "a"}, {"text": "b"}])) == ["one", "two"])
check("the whole chain is a runnable: it has invoke, batch and stream",
      lambda: all(hasattr(summariser(fake("x")), m) for m in ("invoke", "batch", "stream")))

## Section 2 &mdash; Tools: a function plus a docstring

In Module 4 you wrote each tool description by hand, in `TOOL_SPECS`. The `@tool` decorator writes
it for you from three things: the **function name** becomes the tool name, the **type hints** become
the parameters, and the **docstring** becomes the description the model reads.

`open_incident` is left out on purpose. It writes, so it needs a person's approval first. Lab 5.3
adds it safely.

In [ ]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

def make_tools():
    """The three AskOps READ tools, as LangChain tools."""

    @tool
    def search_runbooks(query: str, service: str | None = None) -> str:
        """Find runbooks that match a symptom, such as '502 after deploy'.
        Use for how-to-fix questions. Not for open incidents."""
        return askops.search_runbooks(query, service)

    @tool
    def get_runbook(runbook_id: BLANK) -> str:     # TODO: the type of runbook_id, such as 'RB-101'
        """Return the steps of one runbook by id, such as 'RB-101'.
        Use after search_runbooks. Not for searching."""
        return askops.get_runbook(runbook_id)

    @tool
    def list_incidents() -> str:
        """List the open incidents, newest first."""
        return askops.list_incidents()

    return [search_runbooks, get_runbook, list_incidents]

In [ ]:
# --- Self-check: Section 2
def _schema(name):
    return next(convert_to_openai_tool(t)["function"] for t in make_tools() if t.name == name)

check("three tools, and open_incident is not one of them",
      lambda: [t.name for t in make_tools()] == ["search_runbooks", "get_runbook", "list_incidents"])
check("the function name became the tool name", lambda: _schema("get_runbook")["name"] == "get_runbook")
check("the type hint became a string parameter",
      lambda: _schema("get_runbook")["parameters"]["properties"]["runbook_id"]["type"] == "string",
      "with no type hint, the model has to guess what to send")
check("runbook_id is required", lambda: _schema("get_runbook")["parameters"]["required"] == ["runbook_id"])
check("the docstring became the description",
      lambda: "Not for searching" in _schema("get_runbook")["description"])
check("a parameter with a default value is optional",
      lambda: _schema("search_runbooks")["parameters"]["required"] == ["query"])
check("the tool still runs the Module 4 function",
      lambda: "RB-101" in next(t for t in make_tools() if t.name == "get_runbook").invoke({"runbook_id": "RB-101"}))

In [ ]:
# Read it: what the model receives for get_runbook. Compare it with TOOL_SPECS in agent.py.
guard(lambda: print(json.dumps(_schema("get_runbook"), indent=2)))

## Section 3 &mdash; An agent in one call

`create_agent` runs the whole Module 4 loop. The model asks for a tool, the tool runs, the result goes
back into the messages, and the loop ends when the model replies with no tool call.

Your `MAX_STEPS` is now a setting called `recursion_limit`. It counts **graph steps**, not tool
calls. Each tool call costs 2 steps: the model asks, then the tool runs. Starting and answering cost
2 more. So an agent that may make **N** tool calls needs a limit of **2 &times; N + 2**. When the
limit is reached, the agent raises `GraphRecursionError`.

In [ ]:
from langchain.agents import create_agent
from langgraph.errors import GraphRecursionError

SYSTEM = ("You are AskOps, an assistant for on-call engineers. Answer only from tool results, "
          "in at most 6 lines. Cite runbook ids. If no tool helps, say so.")

def build_agent(model):
    """The AskOps agent: the Module 4 loop, built by LangChain."""
    return create_agent(model, tools=BLANK, system_prompt=SYSTEM)   # TODO: the tools from Section 2

def step_limit(tool_calls):
    """The recursion_limit that lets the agent make this many tool calls, and then answer."""
    return BLANK        # TODO: use the rule above

In [ ]:
def trace(result):
    """Print a run the way agent.py did: one ACTION line per tool call, then steps and tokens."""
    steps = tokens = 0
    for m in result["messages"]:
        if m.type == "ai":
            steps += 1
            tokens += (m.usage_metadata or {}).get("total_tokens", 0)
            for c in m.tool_calls:
                print(f"  step {steps}  ACTION {c['name']}({c['args']})")
    print(f"\n{result['messages'][-1].content}\n\n[{steps} steps, {tokens} tokens]")

QUESTION = "Payments is returning 502s since the 14:00 release. What do I do?"

def _run(model, limit):
    return build_agent(model).invoke({"messages": [("user", QUESTION)]}, {"recursion_limit": limit})

def _happy():
    return _run(fake(call("search_runbooks", 1, query="502 after deploy"),
                     call("get_runbook", 2, runbook_id="RB-101"),
                     "Follow RB-101: check the last release, then roll back."), 25)

def _three_calls(limit):
    """A fake model that makes three tool calls, then answers."""
    try:
        _run(fake(call("list_incidents", 1), call("list_incidents", 2),
                  call("list_incidents", 3), "done"), limit)
        return "finished"
    except GraphRecursionError:
        return "stopped"

def _loops(limit):
    """A fake model that asks for list_incidents twenty times in a row."""
    try:
        _run(fake(*[call("list_incidents", i) for i in range(20)]), limit)
        return "finished"
    except GraphRecursionError:
        return "stopped"

# The scripted run, printed like Day 1. The fake model reports no tokens, so it shows 0.
guard(lambda: trace(_happy()))

In [ ]:
# --- Self-check: Section 3
check("the agent ran both tools and answered",
      lambda: _happy()["messages"][-1].content.startswith("Follow RB-101"))
check("each tool result went back to the model as a message",
      lambda: sum(m.type == "tool" for m in _happy()["messages"]) == 2)
check("the runbook steps reached the model",
      lambda: any(m.type == "tool" and "Roll back" in m.content for m in _happy()["messages"]))
check("step_limit(3) lets three tool calls finish", lambda: _three_calls(step_limit(3)) == "finished")
check("step_limit(3) stops an agent that keeps calling tools",
      lambda: _loops(step_limit(3)) == "stopped",
      "without a limit, a confused model keeps spending tokens")
check("step_limit(3) is the smallest limit that works",
      lambda: _three_calls(step_limit(3) - 1) == "stopped",
      "one step less should stop the same three-call run")

## Run it for real

The same agent on the sandbox model, with the questions from Lab 4.4. Then, in a terminal, run your
Day 1 agent on the same questions (see the lab guide) and compare the ACTION lines, steps and tokens.

In [ ]:
QUESTIONS = [QUESTION, "Logins are slow this morning. Is there a runbook?"]

if llm_ready():
    try:
        agent = build_agent(get_llm())
        for q in QUESTIONS:
            print("Q:", q)
            try:
                trace(agent.invoke({"messages": [("user", q)]}, {"recursion_limit": step_limit(5)}))
            except GraphRecursionError:
                print("Stopped: the step limit was reached.")
            print("-" * 70)
    except NameError:
        print("(fill in the blanks above, then run this cell again)")
    except Exception as exc:
        print(f"<agent run failed: {type(exc).__name__}: {exc}>")

### Streaming

`invoke` shows nothing until the end. `stream` with `stream_mode="messages"` gives you the answer
piece by piece while it is written, which is what a chat window needs.

In [ ]:
if llm_ready():
    try:
        agent = build_agent(get_llm())
        for token, meta in agent.stream({"messages": [("user", QUESTIONS[1])]},
                                        {"recursion_limit": step_limit(5)}, stream_mode="messages"):
            if meta["langgraph_node"] == "tools":
                print(f"\n[tool {token.name} returned {len(token.content)} characters]")
            elif token.content:
                print(token.content, end="", flush=True)
        print()
    except NameError:
        print("(fill in the blanks above, then run this cell again)")
    except Exception as exc:
        print(f"<stream failed: {type(exc).__name__}: {exc}>")

### Read it

Look at what moved into LangChain: the loop, the message list, the tool schemas and the step
budget. Look at what stayed yours: **which tools exist**, **what their docstrings say**, and **which
tools may write**. The framework saves typing. It does not decide what your agent may do.

The tokens should be close to your Day 1 numbers, because the same messages go to the model.
If a run chose different tools from last time, that is normal: a real model picks its path at
run time.

In [ ]:
score()

## Your turn

1. Change the docstring of `get_runbook` to `"Gets data."`, run **Run it for real** again, and compare
   the ACTION lines. Put the docstring back afterwards. It is the same test as Lab 4.4, Step 4.
2. Run the first question with `step_limit(1)`. What does the engineer see? Write the message your
   agent should show instead of a stack trace.